In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import csv
import copy
import pandas as pd
import urllib3
import os
from datetime import datetime
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [3]:
WRITE_PATH = "/content/drive/My Drive/Dugwell/data/yearly/timeseries/"
# WRITE_PATH = "/content/drive/My Drive/Landscape level impact assessment/mani_sarthak/final/dataScrapping/data/timeSeries/"
DIRECTORY_PATH = "/content/drive/My Drive/Dugwell/data/yearly/"

In [4]:
def getYears(path):
    items = os.listdir(path)
    directories = [int(item) for item in items if os.path.isdir(os.path.join(DIRECTORY_PATH, item)) and item[0] != '.' and item != 'check']
    directories.sort()
    return directories

In [5]:
years = getYears(DIRECTORY_PATH)
START_YEAR = years[0]
END_YEAR = years[-1]

# Helper Functions

In [6]:
## just to oopen and read the specified execel file and then return the data in a list of list format
def readFromExcel(state, district, year, season, directory_path = DIRECTORY_PATH):
    excel_path = directory_path + f"{year}/{state}/{district}/{district}_{state}_{year}_{season}.xlsx"
    print(excel_path)
    out = []
    try:
        df = pd.read_excel(excel_path, header=None)
        out = df.values.tolist()[1:]
    except Exception as e:
        print("Error --- \n", year, state, district, season)
        print(e)
    return out

In [7]:
# in order to never miss a data point that is present in the yearly data
# we first iterate over all the seasons in that district for that year.
# once we get the list of all possible stations in a dictionary
# we begin to merge the data in the second loop

def getYearlyDataForDistricts(year, state, district):
    """
    Returns a dictionary containing the station_id as
    key and a list of station_name, lat, long, water_level
    for each season as value.

    None means data not available for that well in that season
    """
    data = dict()
    temp_data = []
    for season in range(1, 5):
        scam = readFromExcel(state, district, year, season)
        print(year, state, district, season, 'done')
        # print(scam, end = "\n\n\n\n")
        temp_data.append(scam)
        for x in scam:
            station_id = x[1]
            station_name = x[0]
            strtion_latitude = x[3]
            station_longitude = x[4]
            water_level = x[2]
            if station_id not in data:
                data[station_id] = [station_name, strtion_latitude, station_longitude]
            elif data[station_id][0] !=  station_name:
                print("some big error in the website or data fetching !!   check data")
                print(year, state, district, season)

    for seasonally in temp_data:
        data_cpy = copy.deepcopy(data)
        for x in seasonally:
            station_id = x[1]
            water_level = x[2]
            if station_id in data:
                data[station_id].append(water_level)
                data_cpy.pop(station_id)
        for x in data_cpy:
            ## None is for empty cell !
            data[x].append(None)
    print(year, state, district, 'done\n\n')
    return data

In [8]:
def mergeThroughYears(state, district, start_year=2010, end_year=2012)   :
    data = dict()
    data['station_id'] = ['station_name', 'latitude', 'longitude']
    temp_data = []
    for year in range(start_year, end_year+1):
        yearly_data_dict = getYearlyDataForDistricts(year, state, district)
        temp_data.append(yearly_data_dict)
        for x in yearly_data_dict:
            station_id = x
            station_name = yearly_data_dict[x][0]
            strtion_latitude = yearly_data_dict[x][1]
            station_longitude = yearly_data_dict[x][2]
            if station_id not in data:
                data[station_id] = [station_name, strtion_latitude, station_longitude]
            elif data[station_id][0] !=  station_name:
                print("some big error in the website or data fetching !!   check data")
                print(year, state, district)
                break
    year = start_year
    for yearly in temp_data:
        data_cpy = copy.deepcopy(data)
        for season in range(1, 5):
            data['station_id'].append(f"{year}_season{season}")
        for x in yearly:
            station_id = x
            water_level = yearly[x][3:]
            season += 1
            if station_id in data:
                data[station_id] += water_level
                data_cpy.pop(station_id)
        for x in data_cpy:
            if x != 'station_id':
                data[x] += ([None]*4)
        year+=1
    print(state, district, f'done from {start_year} to {end_year}\n\n\n\n')
    return data

In [9]:
def modifyDataForCSV(data):
    final_dict = dict()
    final_dict['station_id'] = []
    idx_dict = dict()
    i = 0
    for k in data['station_id']:
        idx_dict[i] = k
        i += 1
        final_dict[k] = []
    for k in data:
        if k == 'station_id':
            continue
        final_dict['station_id'].append(k)
        for i in range(len(data[k])):
            final_dict[idx_dict[i]].append(data[k][i])
    return final_dict

In [10]:
def writeInCSV(data, location, state, district):
    location += f"{state}/{district}/"
    csv_file = location + f"{state}_{district}.csv"
    if not os.path.exists(location):
        os.makedirs(location)
    with open(csv_file, 'w', newline='') as csvfile:
        fieldnames = data.keys()
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # Write the header
        writer.writeheader()

        # Write the dict
        for i in range(len(data['station_id'])):
            row = {field: data[field][i] for field in fieldnames}
            writer.writerow(row)

    print(f'Data saved to {csv_file}')

# Main Functions

In [11]:
def getStates(year):
    path = DIRECTORY_PATH + str(year)+'/'
    items = os.listdir(path)
    directories = [item for item in items if os.path.isdir(os.path.join(path, item)) ]
    directories.sort()
    return directories

states = getStates(years[-1])

In [12]:
def getDistricts(state, year):
    path = DIRECTORY_PATH + str(year)+'/'+str(state)+'/'
    items = os.listdir(path)
    directories = [item for item in items if os.path.isdir(os.path.join(path, item)) ]
    directories.sort()
    return directories

In [13]:
for state in states:
    print(f"starting state: {state}")
    districts = getDistricts(state, END_YEAR)
    for district in districts:
        data = mergeThroughYears(state, district, START_YEAR, END_YEAR)
        data = modifyDataForCSV(data)
        writeInCSV(data, WRITE_PATH, state, district)

Streaming output truncated to the last 5000 lines.




Data saved to /content/drive/My Drive/Dugwell/data/yearly/timeseries/Kerala/WAYANAD/Kerala_WAYANAD.csv
starting state: Madhya Pradesh
/content/drive/My Drive/Dugwell/data/yearly/2025/Madhya Pradesh/AGAR MALWA/AGAR MALWA_Madhya Pradesh_2025_1.xlsx
2025 Madhya Pradesh AGAR MALWA 1 done
/content/drive/My Drive/Dugwell/data/yearly/2025/Madhya Pradesh/AGAR MALWA/AGAR MALWA_Madhya Pradesh_2025_2.xlsx
2025 Madhya Pradesh AGAR MALWA 2 done
/content/drive/My Drive/Dugwell/data/yearly/2025/Madhya Pradesh/AGAR MALWA/AGAR MALWA_Madhya Pradesh_2025_3.xlsx
2025 Madhya Pradesh AGAR MALWA 3 done
/content/drive/My Drive/Dugwell/data/yearly/2025/Madhya Pradesh/AGAR MALWA/AGAR MALWA_Madhya Pradesh_2025_4.xlsx
2025 Madhya Pradesh AGAR MALWA 4 done
2025 Madhya Pradesh AGAR MALWA done


Madhya Pradesh AGAR MALWA done from 2025 to 2025




Data saved to /content/drive/My Drive/Dugwell/data/yearly/timeseries/Madhya Pradesh/AGAR MALWA/Madhya Pradesh_AGAR M